# IEEE39 inverter active-power step: native, ME, and CS

This notebook compares the three-phase EMT IEEE39 bus-30 averaged inverter implementations. The event is a converter command step at 0.1 s, chosen to make the inverter response visible. It is a bounded-error comparison: the present case does not show a new post-event CS divergence.

## Explicit settings

| Setting | Value |
|---|---:|
| Event | Active-power command 125 → 250 MW |
| Event time | 0.1 s |
| Reactive-power command | 80.881 Mvar, constant |
| Simulation stop | 0.3 s |
| Network/communication steps | 50 µs, 1 ms, 5 ms |
| Cases | Native DPsim, FMI Model Exchange, FMI Co-Simulation with explicit zero-order hold |
| Converter current-response lag | 5 ms |
| Current limit | 1.20 pu |
| Initialization | Same IEEE39 power-flow voltages and same inverter initial current in all variants |
| Logged initial current RMS | 237.313923 A in all variants |

The FMU is dual-interface and represents the same averaged inverter equations in ME and CS. Native DPsim uses the matching native averaged-inverter implementation.

## Current response around the event

![One-cycle RMS current envelope for the matched cases](results/power_step/power_step_current_envelope.png)

In [ ]:
from pathlib import Path
import pandas as pd

results = Path("results/power_step")
errors = pd.read_csv(results / "error_summary.csv")
initialization = pd.read_csv(results / "initialization_and_response.csv")
display(initialization)
display(
    errors.pivot(
        index=["step_us", "window"],
        columns="candidate",
        values="relative_error_percent",
    ).round(3)
)

## Interpretation

At 50 µs, the ME inverter current rises from about 111 A immediately before the command to about 196 A after settling. The event is therefore visible at the converter terminal. At 5 ms, explicit CS follows a different current trajectory, but the large error is present before the event: current RMS error relative to ME is 130.94% pre-event, 130.97% during the response, and 130.95% afterward. The trajectories remain finite and settle. This is a large bounded coupling error, not evidence that the event itself causes divergence. At 1 ms, the corresponding errors are approximately 19.2% in each window.

The early current envelope also includes the common initialization transient from the specified initial current to the periodic operating waveform. Since all cases start with the same voltage and current, this startup does not create a comparative initial-condition mismatch; the step-size-dependent separation is a numerical coupling effect.

## Reproduction command

From the repository root, build `FMI_IEEE39_AvgInverter`, then run each mode with the same arguments, changing only the first argument:

```sh
build-fmi/dpsim-fmi/FMI_IEEE39_AvgInverter \
  cs-explicit build-fmi/ieee39-inverter-fmu/IEEE39AvgInverter.fmu \
  0.3 0.005 0.1 30 power-step 250 0
```

Replace `cs-explicit` with `native` or `me` for the other variants. The runner writes the detailed raw CSV traces below `build-fmi/ieee39-inverter/power-step-cosim-demo`.